In [19]:
# Notebook cell: improved extraction + tokenization for your Sentence.pdf / Sentence.docx
# Run in VSCode notebook. If packages not installed, uncomment the pip lines and run once.

# !pip install pdfminer.six python-docx
# (You might already have them installed)

from pdfminer.high_level import extract_text
from docx import Document
import re
import json
import csv
from pathlib import Path

# ---- Configuration ----
# Choose the file you want to process (you uploaded both; change as needed)
# Use raw strings for Windows paths or use Path for cross-platform
pdf_path = Path("docs\Sentence.pdf")
docx_path = Path("docs\Sentence.docx")

# Prefer PDF if it exists, otherwise DOCX
if pdf_path.exists():
    input_path = pdf_path
elif docx_path.exists():
    input_path = docx_path
else:
    raise FileNotFoundError("Neither /mnt/data/Sentence.pdf nor /mnt/data/Sentence.docx was found.")

print("Processing file:", input_path)

# ---- Helpers ----
def read_pdf(path: Path) -> str:
    text = extract_text(str(path))
    return text

def read_docx(path: Path) -> str:
    doc = Document(str(path))
    # join paragraphs with newlines to approximate original layout
    paragraphs = [p.text for p in doc.paragraphs]
    return "\n".join(paragraphs)

def normalize_spaces(s: str) -> str:
    # collapse multiple spaces and normalize newlines
    s = re.sub(r'\r\n?', '\n', s)
    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r'\n{3,}', '\n\n', s)
    return s.strip()

# Sinhala Unicode block: U+0D80 - U+0DFF
SINHALA_REGEX = re.compile(r'[\u0D80-\u0DFF]+')  # captures sequences of Sinhala letters/diacritics

# ---- Read input ----
raw_text = read_pdf(input_path) if input_path.suffix.lower() == ".pdf" else read_docx(input_path)
raw_text = normalize_spaces(raw_text)
print("\n--- Preview of extracted text (first 600 chars) ---\n")
print(raw_text[:600])

# ---- Splitting logic ----
# Strategy A: split by "ක්‍රියාකාරකම" sections (if present)
if "ක්‍රියාකාරකම" in raw_text:
    print("\nDetected 'ක්‍රියාකාරකම' in text -> using that split strategy.")
    # capture title + content blocks
    # this will capture headings like "ක්‍රියාකාරකම-1" or "ක්‍රියාකාරකම 1"
    splits = re.split(r"(ක්‍රියාකාරකම[-–\s]*\d+)", raw_text)
    # splits yields list like: [preface, title1, body1, title2, body2, ...]
    sections = {}
    # iterate pairs
    for i in range(1, len(splits), 2):
        title = splits[i].strip()
        body = normalize_spaces(splits[i+1])
        sections[title] = body

else:
    # Strategy B: split by numeric item markers (e.g., lines starting with 01, 02, 1., 2.)
    print("\nNo 'ක්‍රියාකාරකම' found -> using numeric-item split strategy (01, 02, etc.).")
    # Normalize lines to ensure numbering is at start of lines when possible
    # We'll split using a regex that finds lines beginning with 1- or 2-digit numbers
    # Keep multi-line blocks belonging to each number.
    # Example patterns seen: "01", "02", possibly followed by newline then sentences.
    pattern = re.compile(r'(^\s*\d{1,2}\b.*?$)', flags=re.MULTILINE)
    # find positions of numbered headings
    matches = list(pattern.finditer(raw_text))
    sections = {}
    if matches:
        for idx, m in enumerate(matches):
            start = m.start()
            # end is start of next match or end of text
            end = matches[idx+1].start() if idx+1 < len(matches) else len(raw_text)
            title_line = m.group(1).strip()
            body = raw_text[start:end].strip()
            # Sometimes the title_line is just the number; we'll use it as key with number only
            # But to provide nicer keys, we'll use the number (like "01") as the title
            num_match = re.match(r'^\s*(\d{1,2})\b', title_line)
            title = f"Item-{num_match.group(1).zfill(2)}" if num_match else title_line
            # Remove the leading number from body text for clarity
            body_clean = re.sub(r'^\s*\d{1,2}\b[.\)\s-]*', '', body)
            sections[title] = normalize_spaces(body_clean)
    else:
        # fallback: whole document as single section
        sections = {"full_text": raw_text}

# ---- Tokenize Sinhala words inside each section ----
tokenized_sections = {}
for title, body in sections.items():
    tokens = SINHALA_REGEX.findall(body)
    tokenized_sections[title] = tokens

# ---- Try to parse sentence pairs (many of your example files contain pairs per item) ----
# We'll attempt to find lines and pair them when there are two lines per item as in your example.
pairs = []  # list of tuples (key, left_sentence, right_sentence)
for title, body in sections.items():
    # split into non-empty lines
    lines = [ln.strip() for ln in body.splitlines() if ln.strip()]
    # If the item looks like two parallel lines per number (common in your doc), pair them
    if len(lines) >= 2:
        # heuristic: if even number of lines -> pair sequentially, else pair first two and keep others
        if len(lines) % 2 == 0:
            for i in range(0, len(lines), 2):
                pairs.append((title, lines[i], lines[i+1]))
        else:
            # If odd, try pairing first two, and the rest as singletons
            pairs.append((title, lines[0], lines[1] if len(lines) > 1 else ""))
            for i in range(2, len(lines), 2):
                left = lines[i]
                right = lines[i+1] if i+1 < len(lines) else ""
                pairs.append((title, left, right))
    else:
        # not enough lines to make pairs; skip or keep single
        if lines:
            pairs.append((title, lines[0], ""))

# ---- Output results ----
# 1) Print tokenization summary for first few sections
print("\n--- Tokenization preview (first 6 sections) ---")
for i, (title, tokens) in enumerate(tokenized_sections.items()):
    if i >= 6: break
    print(f"\n{title} -> {len(tokens)} tokens; sample tokens:", tokens[:30])

# 2) Save tokenized sections to JSON
out_json = Path("tokenized_sections.json")
with out_json.open("w", encoding="utf-8") as f:
    json.dump(tokenized_sections, f, ensure_ascii=False, indent=2)
print("\nTokenized sections written to:", out_json.resolve())

# 3) Save paired sentences to CSV for easy viewing in spreadsheet
out_csv = Path("sentence_pairs.csv")
with out_csv.open("w", encoding="utf-8", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["section_key", "left_sentence", "right_sentence"])
    for sec, left, right in pairs:
        writer.writerow([sec, left, right])
print("Sentence pairs written to:", out_csv.resolve())

# 4) Optionally print first 10 pairs to console
print("\n--- First 10 sentence pairs (if any) ---")
for i, (sec, left, right) in enumerate(pairs[:10]):
    print(f"{i+1}. [{sec}]")
    print("  L:", left)
    print("  R:", right)

# ---- Done ----
print("\nProcessing finished. Check tokenized_sections.json and sentence_pairs.csv in your working folder.")


Processing file: docs\Sentence.pdf

--- Preview of extracted text (first 600 chars) ---

01 

02 

03 

04 

05 

06 

07 

ඔහු ගස ේ මල කැඩුවා 

ඔහු ගස ේ මුල කැඩුවා 

නංගී කජු කන්න ආ යි 

නංගී සේජු කන්න ආ යි 

එතන වලේ තියනවා 

එතන විලේ තියනවා 

සගෝනිසේ හාල් තිසයනවා 

සගෝනිසේ හිල් තිසයනවා 

තාත්තා කුඩයේ සගනවා 

තාත්තා කූඩයේ සගනවා 

ගිය මාස ේ කඩපු තැඹිලි ටික පල් 
සවලා 

ගිය මාස ේ කඩපු තැඹිලි ටික 
සපාල් සවලා 

මල්ී බල්ලාව දැේකාද? 

මල්ී බිල්ලාව දැේකාද?

No 'ක්‍රියාකාරකම' found -> using numeric-item split strategy (01, 02, etc.).

--- Tokenization preview (first 6 sections) ---

Item-01 -> 0 tokens; sample tokens: []

Item-02 -> 0 tokens; sample tokens: []

Item-03 -> 0 tokens; sample tokens: []

Item-04 -> 0 tokens; sample tokens: []

Item-05 -> 0 tokens; sample tokens: []

Item-06 -> 0 tokens; sample tokens: []

Tokenized sections written to: C:\Users\94772\Desktop\HearingProject\tokenized_sections.json
Sentence pairs written to: C:\Users\94772\Desktop\HearingProject\sentence_pairs.csv



In [21]:
# === Imports and helpers (your base) ===
import re
import json
from pathlib import Path
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table
from IPython.display import JSON, display

# Regex patterns (no "ක්‍රියාකාරකම" headers — only numbers)
NUMBER_RE = re.compile(r"^(\d{1,2})\b(?:[.)\-]?\s*)(.*)$")
SINHALA_TOKEN_RE = re.compile(r"[\u0D80-\u0DFF]+(?:[\-–—]?[\u0D80-\u0DFF]+)*")

def iter_doc_blocks(doc: Document):
    """Yield paragraphs & table-cell lines (strip empty)."""
    body = doc.element.body
    for child in body.iterchildren():
        if child.tag.endswith('}p'):
            para = Paragraph(child, doc)
            text = para.text.strip()
            if text:
                yield text
        elif child.tag.endswith('}tbl'):
            tbl = Table(child, doc)
            for r in tbl.rows:
                for c in r.cells:
                    for line in c.text.splitlines():
                        line = line.strip()
                        if line:
                            yield line

def extract_tokens_from_text(text):
    """Return Sinhala tokens from text."""
    return [m.group(0) for m in SINHALA_TOKEN_RE.finditer(text)]

# === Read DOCX (preferred) ===
docx_path = Path("docs/Sentence.docx")
doc = Document(str(docx_path))
blocks = list(iter_doc_blocks(doc))

# === Group into numbered sections ===
sections = {}
current = None
for b in blocks:
    m = NUMBER_RE.match(b)
    if m:
        key = f"Item-{m.group(1).zfill(2)}"
        rest = m.group(2).strip()
        sections[key] = [rest] if rest else []
        current = key
    elif current:
        sections[current].append(b)

sections_text = {k: "\n".join(v).strip() for k, v in sections.items()}

# === Tokenize Sinhala text ===
tokenized_sections = {k: extract_tokens_from_text(v) for k, v in sections_text.items()}

# === Extract sentence pairs (each item has 2 lines) ===
pairs = []
for k, v in sections_text.items():
    lines = [ln.strip() for ln in v.splitlines() if ln.strip()]
    for i in range(0, len(lines), 2):
        left = lines[i]
        right = lines[i+1] if i+1 < len(lines) else ""
        pairs.append((k, left, right))

# === Save outputs ===
Path("tokenized_sections.json").write_text(
    json.dumps(tokenized_sections, ensure_ascii=False, indent=2), encoding="utf-8"
)

import csv
with open("sentence_pairs.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["section_key", "left_sentence", "right_sentence"])
    writer.writerows(pairs)

# === Quick preview ===
print(f"Total items: {len(sections)}")
for k, v in list(tokenized_sections.items())[:3]:
    print(f"\n{k}: {len(v)} tokens\n", v[:20])

display(JSON(tokenized_sections))


Total items: 7

Item-01: 8 tokens
 ['ඔහු', 'ගසේ', 'මල', 'කැඩුවා', 'ඔහු', 'ගසේ', 'මුල', 'කැඩුවා']

Item-02: 8 tokens
 ['නංගී', 'කජු', 'කන්න', 'ආසයි', 'නංගී', 'කේජු', 'කන්න', 'ආසයි']

Item-03: 6 tokens
 ['එතන', 'වලක්', 'තියනවා', 'එතන', 'විලක්', 'තියනවා']


<IPython.core.display.JSON object>

In [22]:
def parse_blocks(blocks):
    """
    Parse a list of ordered text blocks (paragraphs / table-cell lines)
    into a simple numbered structure suitable for your files.

    Behavior:
    - Recognizes lines like "01" or "01 <text>" or "1." as item starts.
    - Collects up to two following non-number lines as the paired sentences.
    - Extracts Sinhala tokens (via extract_tokens_from_text) and stores up to 2 tokens per item.
    - Returns: { "NoHeader": { "items": [ { "index": "01", "lines": [...], "words": [...] }, ...],
                               "tokens": [...] } }
    """
    result = {"NoHeader": {"items": [], "tokens": []}}
    NUMBER_LINE_RE = re.compile(r"^\s*(\d{1,2})\b(?:[.)\-]?\s*)(.*)$")  # captures number and optional rest
    NUMBER_ONLY_RE = re.compile(r"^\s*(\d{1,2})\s*\.?$")

    i = 0
    n = len(blocks)
    while i < n:
        b = blocks[i].strip()
        if not b:
            i += 1
            continue

        m = NUMBER_LINE_RE.match(b)
        if m:
            idx = m.group(1).zfill(2)
            rest = m.group(2).strip()
            collected_lines = []
            # If the number line already contains text after the number, use it first
            if rest:
                collected_lines.append(rest)

            # Collect up to 2 non-number lines following this marker
            j = i + 1
            while j < n and len(collected_lines) < 2:
                nxt = blocks[j].strip()
                if NUMBER_ONLY_RE.match(nxt):  # next entry starts
                    break
                if nxt:
                    collected_lines.append(nxt)
                j += 1

            # Extract Sinhala tokens (fall back to raw line if no tokens)
            words = []
            for line in collected_lines:
                toks = extract_tokens_from_text(line)
                if toks:
                    words.extend(toks)
                else:
                    words.append(line)

            words = words[:2]  # keep at most 2 tokens/words per item

            result["NoHeader"]["items"].append({
                "index": idx,
                "lines": collected_lines,
                "words": words
            })
            for w in words:
                result["NoHeader"]["tokens"].append(w)

            # advance i to j (start of next candidate)
            i = j
            continue

        # In case file uses a bare number line like "01" (caught above) — otherwise skip stray lines
        i += 1

    return result


In [23]:
# Main execution cell — concise and robust
import json
from pathlib import Path
from docx import Document
from IPython.display import JSON, display

DOCX_PATH = Path("docs/Sentence.docx")   # <- points to your uploaded file
OUT_JSON = Path("sentence_from_docx_notebook.json")

if not DOCX_PATH.exists():
    raise FileNotFoundError(f"{DOCX_PATH} not found. Put Sentence.docx in /mnt/data or change DOCX_PATH.")

# load doc and collect non-empty blocks using your iter_doc_blocks()
doc = Document(str(DOCX_PATH))
blocks = [b for b in iter_doc_blocks(doc) if b.strip()]

print(f"Found {len(blocks)} text blocks (paragraph lines & table cells).")

# parse (uses your parse_blocks function)
parsed = parse_blocks(blocks)

if not parsed:
    print("No sections parsed. Showing first 50 blocks for debugging:")
    for k, b in enumerate(blocks[:50], start=1):
        print(f"{k:02d} | {b!r}")
else:
    # save JSON with fallback
    try:
        OUT_JSON.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Saved JSON to:", OUT_JSON.resolve())
    except Exception as e:
        print("Primary save failed:", repr(e))
        try:
            fallback = Path("sentence_from_docx_notebook_fallback.json")
            fallback.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
            print("Fallback saved to:", fallback.resolve())
        except Exception as e2:
            print("Fallback save also failed:", repr(e2))

    # display preview of first section
    first_key = next(iter(parsed))
    print("\nPreview (first section):", first_key)
    display(JSON(parsed[first_key]))


Found 21 text blocks (paragraph lines & table cells).
Saved JSON to: C:\Users\94772\Desktop\HearingProject\sentence_from_docx_notebook.json

Preview (first section): NoHeader


<IPython.core.display.JSON object>

In [25]:
# Main execution cell — concise and robust
import json
from pathlib import Path
from docx import Document
from IPython.display import JSON, display

DOCX_PATH = Path("docs/Sentence.docx")   # <- points to your uploaded file
OUT_JSON = Path("sentence_from_docx_notebook.json")

if not DOCX_PATH.exists():
    raise FileNotFoundError(f"{DOCX_PATH} not found. Put Sentence.docx in /mnt/data or change DOCX_PATH.")

# load doc and collect non-empty blocks using your iter_doc_blocks()
doc = Document(str(DOCX_PATH))
blocks = [b for b in iter_doc_blocks(doc) if b.strip()]

print(f"Found {len(blocks)} text blocks (paragraph lines & table cells).")

# parse (uses your parse_blocks function)
parsed = parse_blocks(blocks)

if not parsed:
    print("No sections parsed. Showing first 50 blocks for debugging:")
    for k, b in enumerate(blocks[:50], start=1):
        print(f"{k:02d} | {b!r}")
else:
    # save JSON with fallback
    try:
        OUT_JSON.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Saved JSON to:", OUT_JSON.resolve())
    except Exception as e:
        print("Primary save failed:", repr(e))
        try:
            fallback = Path("sentence_from_docx_notebook_fallback.json")
            fallback.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
            print("Fallback saved to:", fallback.resolve())
        except Exception as e2:
            print("Fallback save also failed:", repr(e2))

    # display preview of first section
    first_key = next(iter(parsed))
    print("\nPreview (first section):", first_key)
    display(JSON(parsed[first_key]))


Found 21 text blocks (paragraph lines & table cells).
Saved JSON to: C:\Users\94772\Desktop\HearingProject\sentence_from_docx_notebook.json

Preview (first section): NoHeader


<IPython.core.display.JSON object>

In [27]:
# Prepare JSONs: keep full sentences and highlighted words + transliterations
import json
from pathlib import Path
import unicodedata

IN_JSON = Path("sentence_from_docx_notebook.json")   # input from your parse cell
OUT_SINH = Path("sentence_transliterated.json")      # output with Sinhala sentences preserved
OUT_TRANS = Path("sentence_translit_improved.json")  # output with Singlish transliteration

if not IN_JSON.exists():
    raise FileNotFoundError(f"Missing input: {IN_JSON}")

data = json.loads(IN_JSON.read_text(encoding="utf-8"))

# --- transliteration utilities (compact copy of your transliterator) ---
CONS = {"ක":"k","ඛ":"kh","ග":"g","ඝ":"gh","ඟ":"ng","ච":"ch","ඡ":"chh","ජ":"j","ඣ":"jh","ඤ":"ny",
        "ට":"t","ඨ":"th","ඩ":"d","ඪ":"dh","ණ":"n","ත":"t","ථ":"th","ද":"d","ධ":"dh","න":"n",
        "ප":"p","ඵ":"ph","බ":"b","භ":"bh","ම":"m","ය":"y","ර":"r","ල":"l","ව":"v","ශ":"sh",
        "ෂ":"sh","ස":"s","හ":"h","ළ":"l","ෆ":"f","ඥ":"gn","ඦ":"gn"}
INDEP_V = {"අ":"a","ආ":"aa","ඇ":"ae","ඈ":"aae","ඉ":"i","ඊ":"ii","උ":"u","ඌ":"uu",
           "එ":"e","ඒ":"ee","ඔ":"o","ඕ":"oo","ඓ":"ai","ඖ":"au"}
V_SIGN = {"ා":"aa","ි":"i","ී":"ii","ු":"u","ූ":"uu","ෙ":"e","ේ":"ee","ෛ":"ai","ො":"o","ෝ":"oo","ෞ":"au",
          "ෘ":"ru","ෲ":"ruu"}
DIAC = {"ං":"n","ඃ":"h"}
VIRAMA = "්"
INDEP_SET = set(INDEP_V.keys()); CONS_SET = set(CONS.keys())
V_SIGN_SET = set(V_SIGN.keys()); DIAC_SET = set(DIAC.keys())

def normalize_text(s: str) -> str:
    return unicodedata.normalize("NFC", s)

def transliterate_text(s: str, cap_first: bool = False) -> str:
    # transliterate an entire Sinhala string (sentence or word)
    w = normalize_text(s)
    i = 0; n = len(w); out = []
    while i < n:
        ch = w[i]
        if ch in INDEP_SET:
            out.append(INDEP_V[ch]); i += 1; continue
        if ch in CONS_SET:
            # gemination
            if i+2 < n and w[i+1] == VIRAMA and w[i+2] == ch:
                base = CONS[ch]; i += 3
                vowel_part = ""
                while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                    if w[i] in V_SIGN_SET: vowel_part = V_SIGN[w[i]]
                    elif w[i] in DIAC_SET: vowel_part += DIAC[w[i]]
                    i += 1
                if vowel_part == "": vowel_part = "a"
                out.append(base + base + vowel_part); continue
            base = CONS[ch]; i += 1
            if i < n and w[i] == VIRAMA:
                i += 1; out.append(base); continue
            vowel_part = ""
            while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                if w[i] in V_SIGN_SET: vowel_part = V_SIGN[w[i]]
                elif w[i] in DIAC_SET: vowel_part += DIAC[w[i]]
                i += 1
            if vowel_part == "": vowel_part = "a"
            out.append(base + vowel_part); continue
        if ch in V_SIGN_SET:
            out.append(V_SIGN[ch]); i += 1; continue
        if ch in DIAC_SET:
            out.append(DIAC[ch]); i += 1; continue
        # passthrough (spaces, punctuation, Latin etc.)
        out.append(ch); i += 1
    res = "".join(out)
    if cap_first and res:
        res = res[0].upper() + res[1:]
    return res

# --- build outputs ---
out_sinh = {"NoHeader": {"items": [], "tokens": []}}
out_trans = {"NoHeader": {"items": [], "tokens": []}}

for sec, payload in data.items():
    for it in payload.get("items", []):
        idx = it.get("index", "")
        lines = it.get("lines") or it.get("sentences") or []
        # ensure lines is a list of strings
        lines = [str(x) for x in lines]
        highlights = it.get("words") or it.get("highlighted") or []
        highlights = [str(x) for x in highlights]

        # Sinhala-preserving output (full sentences + highlighted words)
        out_item_s = {
            "index": idx,
            "sentences": lines,
            "highlighted_words": highlights
        }
        out_sinh["NoHeader"]["items"].append(out_item_s)
        out_sinh["NoHeader"]["tokens"].extend(highlights)

        # Transliteration output (full sentences transliterated + highlighted translit)
        sent_trans = [transliterate_text(s, cap_first=True) for s in lines]
        high_trans = [transliterate_text(w, cap_first=True) for w in highlights]
        out_item_t = {
            "index": idx,
            "sentences_translit": sent_trans,
            "highlighted_translit": high_trans
        }
        out_trans["NoHeader"]["items"].append(out_item_t)
        out_trans["NoHeader"]["tokens"].extend(high_trans)

# --- save files ---
OUT_SINH.write_text(json.dumps(out_sinh, ensure_ascii=False, indent=2), encoding="utf-8")
OUT_TRANS.write_text(json.dumps(out_trans, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:", OUT_SINH.resolve())
print("Wrote:", OUT_TRANS.resolve())


Wrote: C:\Users\94772\Desktop\HearingProject\sentence_transliterated.json
Wrote: C:\Users\94772\Desktop\HearingProject\sentence_translit_improved.json


In [28]:
# Run-level DOCX parsing: extract lines + run-level highlighted words per block, then parse numbered items
import re, json
from pathlib import Path
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table
from docx.shared import RGBColor

# Regex for Sinhala tokens (same as yours)
SINHALA_TOKEN_RE = re.compile(r"[\u0D80-\u0DFF]+(?:[\-–—]?[\u0D80-\u0DFF]+)*")

# Paths
DOCX_PATH = Path("docs/Sentence.docx")
OUT_JSON = Path("sentence_from_docx_notebook_with_highlights.json")

if not DOCX_PATH.exists():
    raise FileNotFoundError(f"{DOCX_PATH} not found. Put Sentence.docx in /mnt/data or change path.")

def run_is_highlighted(run):
    """
    Heuristic to decide if a run is 'highlighted' in the Word doc:
    - explicit highlight color set (preferred),
    - OR run.bold is True,
    - OR run.font.color.rgb is set and not black (heuristic).
    """
    f = run.font
    try:
        if getattr(f, "highlight_color", None):
            return True
    except Exception:
        # some python-docx versions may behave differently; ignore
        pass
    try:
        if f.bold:
            return True
    except Exception:
        pass
    try:
        col = f.color.rgb
        if col and isinstance(col, RGBColor):
            # if not black (00 00 00) treat as highlighted
            if col != RGBColor(0,0,0):
                return True
    except Exception:
        pass
    return False

def iter_doc_blocks_with_highlights(doc: Document):
    """
    Yield dicts for each paragraph or table-cell line found in document body in order:
      { "text": "full text of block", "highlights": ["word1","word2", ...] }
    Highlights are taken from runs inside that paragraph/cell using run_is_highlighted().
    """
    body = doc.element.body
    for child in body.iterchildren():
        if child.tag.endswith('}p'):
            para = Paragraph(child, doc)
            text = para.text.strip()
            if not text:
                continue
            # collect highlighted text fragments from runs
            highlights = []
            for run in para.runs:
                if not run.text:
                    continue
                if run_is_highlighted(run):
                    # split run text into Sinhala tokens (keep as-is if no tokens found)
                    toks = SINHALA_TOKEN_RE.findall(run.text)
                    if toks:
                        highlights.extend(toks)
                    else:
                        # if run contains punctuation/space or whole words, split by whitespace and extend
                        highlights.extend([t.strip() for t in run.text.split() if t.strip()])
            yield {"text": text, "highlights": highlights}

        elif child.tag.endswith('}tbl'):
            tbl = Table(child, doc)
            for r in tbl.rows:
                for c in r.cells:
                    # iterate lines inside cell keeping run-level highlights per line
                    # note: docx returns cell.text as single string; to get runs, inspect paragraphs in cell
                    for para in c.paragraphs:
                        text = para.text.strip()
                        if not text:
                            continue
                        highlights = []
                        for run in para.runs:
                            if not run.text:
                                continue
                            if run_is_highlighted(run):
                                toks = SINHALA_TOKEN_RE.findall(run.text)
                                if toks:
                                    highlights.extend(toks)
                                else:
                                    highlights.extend([t.strip() for t in run.text.split() if t.strip()])
                        yield {"text": text, "highlights": highlights}

# Build ordered blocks
doc = Document(str(DOCX_PATH))
blocks = list(iter_doc_blocks_with_highlights(doc))
print("Found blocks:", len(blocks))
# Optional quick view:
for i, b in enumerate(blocks[:12], 1):
    print(f"{i:02d} | {b['text'][:80]!r}  HIGHLIGHTS={b['highlights']}")

# Now parse numbered items (01, 02, ...) exactly like before, but preserve highlight lists
NUMBER_LINE_RE = re.compile(r"^\s*(\d{1,2})\b(?:[.)\-]?\s*)(.*)$")
NUMBER_ONLY_RE = re.compile(r"^\s*(\d{1,2})\s*\.?$")

result = {"NoHeader": {"items": [], "tokens": []}}
i = 0
n = len(blocks)

while i < n:
    b = blocks[i]["text"].strip()
    m = NUMBER_LINE_RE.match(b)
    if m:
        idx = m.group(1).zfill(2)
        rest = m.group(2).strip()
        collected_lines = []
        collected_highlights = []
        # if the numbered line contains text after the number, use it
        if rest:
            collected_lines.append(rest)
            # also include highlights from this block (if any)
            collected_highlights.extend(blocks[i].get("highlights", []))
        # collect up to 2 following non-number blocks as sentence lines
        j = i + 1
        while j < n and len(collected_lines) < 2:
            nxt_text = blocks[j]["text"].strip()
            if NUMBER_ONLY_RE.match(nxt_text):
                break
            if nxt_text:
                collected_lines.append(nxt_text)
                collected_highlights.extend(blocks[j].get("highlights", []))
            j += 1

        # If no highlighted runs found, fall back to token-extraction from the first collected line
        if not collected_highlights and collected_lines:
            first_line = collected_lines[0]
            toks = SINHALA_TOKEN_RE.findall(first_line)
            # heuristics: choose first two tokens as highlighted words
            collected_highlights = toks[:2]

        # normalize highlighted list (keep order, unique-ish)
        # often runs produce repeated tokens, keep first occurrences:
        seen = set()
        final_highlights = []
        for h in collected_highlights:
            if h not in seen:
                final_highlights.append(h)
                seen.add(h)
        final_highlights = final_highlights[:2]  # keep at most 2, matching your pipeline

        # store
        result["NoHeader"]["items"].append({
            "index": idx,
            "lines": collected_lines,
            "highlighted_words": final_highlights
        })
        result["NoHeader"]["tokens"].extend(final_highlights)

        i = j
        continue

    # skip stray blocks
    i += 1

# Save result
OUT_JSON.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", OUT_JSON.resolve())


Found blocks: 21
01 | '01'  HIGHLIGHTS=[]
02 | 'ඔහු ගසේ මල කැඩුවා'  HIGHLIGHTS=['මල']
03 | 'ඔහු ගසේ මුල කැඩුවා'  HIGHLIGHTS=['මුල']
04 | '02'  HIGHLIGHTS=[]
05 | 'නංගී කජු කන්න ආසයි'  HIGHLIGHTS=['කජු']
06 | 'නංගී කේජු කන්න ආසයි'  HIGHLIGHTS=['කේජු']
07 | '03'  HIGHLIGHTS=[]
08 | 'එතන වලක් තියනවා'  HIGHLIGHTS=['වලක්']
09 | 'එතන විලක් තියනවා'  HIGHLIGHTS=['විලක්']
10 | '04'  HIGHLIGHTS=[]
11 | 'ගෝනියේ හාල් තියෙනවා'  HIGHLIGHTS=['හාල්']
12 | 'ගෝනියේ හිල් තියෙනවා'  HIGHLIGHTS=['හිල්']
Wrote: C:\Users\94772\Desktop\HearingProject\sentence_from_docx_notebook_with_highlights.json


In [30]:
# Notebook cell: extract first-letter vowel + transliteration for HIGHLIGHTED words from sentences
import re, json, csv, unicodedata
from pathlib import Path

# --- Input / Output file paths ---
# (adjust if you renamed your parsed JSON)
IN_JSON = Path("sentence_from_docx_notebook_with_highlights.json")
if not IN_JSON.exists():
    IN_JSON = Path("sentence_from_docx_notebook.json")  # fallback

OUT_CSV = Path("sentence_highlighted_words_vowels.csv")
OUT_JSON = Path("sentence_highlighted_words_vowels.json")

# --- Sinhala mappings ---
INDEP_VOWELS = {
    "අ":"a","ආ":"aa","ඇ":"ae","ඈ":"aae","ඉ":"i","ඊ":"ii",
    "උ":"u","ඌ":"uu","එ":"e","ඒ":"ee","ඔ":"o","ඕ":"oo",
    "ඓ":"ai","ඖ":"au"
}
VOWEL_SIGN_TO_INDEP = {
    "ා":("ආ","aa"),"ි":("ඉ","i"),"ී":("ඊ","ii"),
    "ු":("උ","u"),"ූ":("ඌ","uu"),"ෙ":("එ","e"),
    "ේ":("ඒ","ee"),"ෛ":("ඓ","ai"),"ො":("ඔ","o"),
    "ෝ":("ඕ","oo"),"ෞ":("ඖ","au"),
    "ෘ":("උ (approx)","ru"),"ෲ":("ඌ (approx)","ruu")
}
CONS = {
    "ක":"k","ඛ":"kh","ග":"g","ඝ":"gh","ඟ":"ng","ච":"ch","ඡ":"chh",
    "ජ":"j","ඣ":"jh","ඤ":"ny","ට":"t","ඨ":"th","ඩ":"d","ඪ":"dh",
    "ණ":"n","ත":"t","ථ":"th","ද":"d","ධ":"dh","න":"n","ප":"p",
    "ඵ":"ph","බ":"b","භ":"bh","ම":"m","ය":"y","ර":"r","ල":"l",
    "ව":"v","ශ":"sh","ෂ":"sh","ස":"s","හ":"h","ළ":"l","ෆ":"f",
    "ඥ":"gn","ඦ":"gn"
}
V_SIGN = {
    "ා":"aa","ි":"i","ී":"ii","ු":"u","ූ":"uu","ෙ":"e",
    "ේ":"ee","ෛ":"ai","ො":"o","ෝ":"oo","ෞ":"au","ෘ":"ru","ෲ":"ruu"
}
DIAC = {"ං":"n","ඃ":"h"}
VIRAMA = "්"

# --- sets for fast checks ---
INDEP_SET = set(INDEP_VOWELS.keys())
SIGN_SET = set(VOWEL_SIGN_TO_INDEP.keys())
CONS_SET = set(CONS.keys())
V_SIGN_SET = set(V_SIGN.keys())
DIAC_SET = set(DIAC.keys())

SINHALA_TOKEN_RE = re.compile(r"[\u0D80-\u0DFF]+(?:[-–—]?[\u0D80-\u0DFF]+)*")

def normalize_text(s: str) -> str:
    return unicodedata.normalize("NFC", s)

def first_letter_vowel(word: str):
    """Return (vowel_char, vowel_name) for the first Sinhala syllable."""
    w = normalize_text(str(word)).strip()
    if not w:
        return "",""
    first = w[0]
    if first in INDEP_SET:
        return first, INDEP_VOWELS[first]
    if '\u0D80' <= first <= '\u0DFF':
        if len(w) >= 2:
            nxt = w[1]
            if nxt in SIGN_SET:
                return VOWEL_SIGN_TO_INDEP[nxt]
            if nxt == VIRAMA:
                return "අ","a"
        return "අ","a"
    return "",""

def transliterate_word(word: str, cap_first: bool = True) -> str:
    """Convert Sinhala → Singlish (keeps punctuation)."""
    w = normalize_text(str(word))
    i, n = 0, len(w)
    out = []
    while i < n:
        ch = w[i]
        if ch in INDEP_SET:
            out.append(INDEP_VOWELS[ch]); i += 1; continue
        if ch in CONS_SET:
            if i+2 < n and w[i+1] == VIRAMA and w[i+2] == ch:
                base = CONS[ch]; i += 3; vowel_part = ""
                while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                    if w[i] in V_SIGN_SET: vowel_part = V_SIGN[w[i]]
                    elif w[i] in DIAC_SET: vowel_part += DIAC[w[i]]
                    i += 1
                if not vowel_part: vowel_part = "a"
                out.append(base + base + vowel_part); continue
            base = CONS[ch]; i += 1
            if i < n and w[i] == VIRAMA:
                i += 1; out.append(base); continue
            vowel_part = ""
            while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                if w[i] in V_SIGN_SET: vowel_part = V_SIGN[w[i]]
                elif w[i] in DIAC_SET: vowel_part += DIAC[w[i]]
                i += 1
            if not vowel_part: vowel_part = "a"
            out.append(base + vowel_part); continue
        if ch in V_SIGN_SET:
            out.append(V_SIGN[ch]); i += 1; continue
        if ch in DIAC_SET:
            out.append(DIAC[ch]); i += 1; continue
        out.append(ch); i += 1
    res = "".join(out)
    if cap_first and res:
        res = res[0].upper() + res[1:]
    return res

# --- Load parsed sentence data ---
if not IN_JSON.exists():
    raise FileNotFoundError(f"Input file not found: {IN_JSON}")
data = json.loads(IN_JSON.read_text(encoding="utf-8"))

rows = []
for sec, payload in data.items():
    for it in payload.get("items", []):
        idx = it.get("index", "")
        sentences = it.get("lines") or it.get("sentences") or []
        sentences = [str(x) for x in sentences]
        highlights = it.get("highlighted_words") or it.get("words") or []
        if not highlights and sentences:
            highlights = SINHALA_TOKEN_RE.findall(sentences[0])[:2]
        highlights = [str(h) for h in highlights][:2]

        for pos, hw in enumerate(highlights, start=1):
            vowel_char, vowel_name = first_letter_vowel(hw)
            singlish = transliterate_word(hw)
            rows.append({
                "section": sec,
                "index": idx,
                "sentence_text": " || ".join(sentences),
                "highlight_pos": pos,
                "highlight_sinhala": hw,
                "highlight_singlish": singlish,
                "main_vowel_char": vowel_char,
                "main_vowel_name": vowel_name
            })

# --- Save outputs ---
with OUT_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "section","index","sentence_text","highlight_pos",
        "highlight_sinhala","highlight_singlish",
        "main_vowel_char","main_vowel_name"
    ])
    writer.writeheader()
    writer.writerows(rows)

OUT_JSON.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:", OUT_CSV.resolve())
print("Wrote:", OUT_JSON.resolve())

# quick preview
for r in rows[:5]:
    print(r)


Wrote: C:\Users\94772\Desktop\HearingProject\sentence_highlighted_words_vowels.csv
Wrote: C:\Users\94772\Desktop\HearingProject\sentence_highlighted_words_vowels.json
{'section': 'NoHeader', 'index': '01', 'sentence_text': 'ඔහු ගසේ මල කැඩුවා || ඔහු ගසේ මුල කැඩුවා', 'highlight_pos': 1, 'highlight_sinhala': 'මල', 'highlight_singlish': 'Mala', 'main_vowel_char': 'අ', 'main_vowel_name': 'a'}
{'section': 'NoHeader', 'index': '01', 'sentence_text': 'ඔහු ගසේ මල කැඩුවා || ඔහු ගසේ මුල කැඩුවා', 'highlight_pos': 2, 'highlight_sinhala': 'මුල', 'highlight_singlish': 'Mula', 'main_vowel_char': 'උ', 'main_vowel_name': 'u'}
{'section': 'NoHeader', 'index': '02', 'sentence_text': 'නංගී කජු කන්න ආසයි || නංගී කේජු කන්න ආසයි', 'highlight_pos': 1, 'highlight_sinhala': 'කජු', 'highlight_singlish': 'Kaju', 'main_vowel_char': 'අ', 'main_vowel_name': 'a'}
{'section': 'NoHeader', 'index': '02', 'sentence_text': 'නංගී කජු කන්න ආසයි || නංගී කේජු කන්න ආසයි', 'highlight_pos': 2, 'highlight_sinhala': 'කේජු', 'highli

In [34]:
# SINGLE NOTEBOOK CELL: Generate audio files for each FULL sentence (one file per sentence)
# Install if needed:
# !pip install google-cloud-texttospeech gTTS pydub tqdm unidecode

import os, json, io, re, csv
from pathlib import Path
from tqdm import tqdm
from unidecode import unidecode
from pydub import AudioSegment

# Try to import Google Cloud TTS and enable only if creds present
USE_GOOGLE_CLOUD = False
try:
    from google.cloud import texttospeech
    if os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
        USE_GOOGLE_CLOUD = True
except Exception:
    USE_GOOGLE_CLOUD = False

from gtts import gTTS

# ---------- Config ----------
# Prefer run-level JSON with highlights (contains 'lines' per item)
INPUT_JSON = Path("sentence_from_docx_notebook_with_highlights.json")
if not INPUT_JSON.exists():
    INPUT_JSON = Path("sentence_from_docx_notebook.json")  # fallback
if not INPUT_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {INPUT_JSON.resolve()}")

OUTPUT_ROOT = Path("hearing_project_v0.01/sentence_audio")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_CSV = OUTPUT_ROOT / "manifest.csv"

LANG = "si"
SLOW_FACTOR = 1.12       # >1 -> slower
VOLUME_GAIN_DB = 4       # volume boost

# GCP settings (only used if USE_GOOGLE_CLOUD True)
GCP_VOICE_NAME = "si-LK-Wavenet-A"
GCP_SPEAKING_RATE = 0.95
GCP_PITCH = 0.0

# Pronunciation overrides: map full sentence -> alternate text for TTS
# e.g. pronunciation_overrides["ඔහු ගසේ මල කැඩුවා"] = "ඔහු ගසේ මල කැ-ඩු-වා"
pronunciation_overrides = {
    # add sentence-specific overrides here
}

# ---------- Helpers ----------
def safe_unicode_filename(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"[\\/:\*\?\"<>\|]", "_", s)
    s = re.sub(r"\s+", "_", s)
    # cut to reasonable length
    return s[:60]

def ascii_snippet(s: str, maxlen=25) -> str:
    s = unidecode(str(s))
    s = re.sub(r"[^\w\s\-]", "", s)
    s = s.strip().replace(" ", "_")
    return (s[:maxlen] or "sentence").strip("_")

def change_speed(sound: AudioSegment, factor: float) -> AudioSegment:
    new_frame_rate = int(sound.frame_rate / factor)
    slowed = sound._spawn(sound.raw_data, overrides={"frame_rate": new_frame_rate})
    return slowed.set_frame_rate(sound.frame_rate)

def save_audiosegment_to_mp3(audio: AudioSegment, path: Path, bitrate="192k"):
    audio.export(str(path), format="mp3", bitrate=bitrate)

# ---------- Load input ----------
data = json.loads(INPUT_JSON.read_text(encoding="utf-8"))

# Flatten items to list with context (section, index, sentences[])
items = []
for sec, payload in data.items():
    for it in payload.get("items", []):
        idx = str(it.get("index", "")).zfill(2)
        # lines / sentences field
        sentences = it.get("lines") or it.get("sentences") or it.get("lines") or []
        # ensure list of strings
        sentences = [str(x).strip() for x in sentences if str(x).strip()]
        if not sentences:
            # fallback: maybe highlighted words exist; skip if nothing
            continue
        items.append({"section": sec, "index": idx, "sentences": sentences})

# Group by section for folder structure (preserve order)
from collections import OrderedDict
sections = OrderedDict()
for it in items:
    sec = it["section"]
    sections.setdefault(sec, []).append(it)

# Init GCP client if requested
gcp_client = None
if USE_GOOGLE_CLOUD:
    try:
        gcp_client = texttospeech.TextToSpeechClient()
    except Exception as e:
        print("GCP init failed, falling back to gTTS:", e)
        USE_GOOGLE_CLOUD = False

# Prepare manifest CSV writer
manifest_rows = []
manifest_header = ["section","index","sentence_no","sentence_text","file_path","method"]

# ---------- Generation loop ----------
total_sentences = sum(len(it["sentences"]) for it in items)
pbar = tqdm(total=total_sentences, desc="Generating sentence audio")

for sec, its in sections.items():
    sec_folder = OUTPUT_ROOT / safe_unicode_filename(sec)
    sec_folder.mkdir(parents=True, exist_ok=True)

    for it in its:
        idx = it["index"]
        sentences = it["sentences"]
        # generate audio for each sentence (1-based numbering)
        for s_no, sentence in enumerate(sentences, start=1):
            # choose text to send to TTS: override if provided
            tts_text = pronunciation_overrides.get(sentence, sentence)

            # create filename snippet (based on sentence or section)
            snippet = ascii_snippet(sentence)
            fname = f"{idx}_{s_no}_{snippet}.mp3"
            out_path = sec_folder / fname

            # skip if file exists
            if out_path.exists():
                manifest_rows.append([sec, idx, s_no, sentence, str(out_path.resolve()), "skipped"])
                pbar.update(1)
                continue

            try:
                if USE_GOOGLE_CLOUD and gcp_client:
                    input_text = texttospeech.SynthesisInput(text=tts_text)
                    voice = texttospeech.VoiceSelectionParams(language_code="si-LK", name=GCP_VOICE_NAME)
                    audio_config = texttospeech.AudioConfig(
                        audio_encoding=texttospeech.AudioEncoding.MP3,
                        speaking_rate=GCP_SPEAKING_RATE,
                        pitch=GCP_PITCH
                    )
                    response = gcp_client.synthesize_speech(input=input_text, voice=voice, audio_config=audio_config)
                    audio_bytes = response.audio_content
                    audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")
                    method = "gcp_wavenet"
                else:
                    tts = gTTS(text=tts_text, lang=LANG, slow=False)
                    buf = io.BytesIO()
                    tts.write_to_fp(buf)
                    buf.seek(0)
                    audio = AudioSegment.from_file(buf, format="mp3")
                    method = "gTTS"

                # post-process
                processed = change_speed(audio, SLOW_FACTOR)
                processed = processed + VOLUME_GAIN_DB
                save_audiosegment_to_mp3(processed, out_path)

                manifest_rows.append([sec, idx, s_no, sentence, str(out_path.resolve()), method])

            except Exception as e:
                print(f"[ERROR] section={sec} index={idx} sentence_no={s_no}: {e}")
                # fallback attempt with gTTS if not already tried
                if method != "gTTS":
                    try:
                        tts = gTTS(text=tts_text, lang=LANG, slow=False)
                        buf = io.BytesIO(); tts.write_to_fp(buf); buf.seek(0)
                        audio = AudioSegment.from_file(buf, format="mp3")
                        processed = change_speed(audio, SLOW_FACTOR)
                        processed = processed + VOLUME_GAIN_DB
                        save_audiosegment_to_mp3(processed, out_path)
                        manifest_rows.append([sec, idx, s_no, sentence, str(out_path.resolve()), "gTTS_fallback"])
                    except Exception as e2:
                        print("Fallback also failed:", e2)
                        manifest_rows.append([sec, idx, s_no, sentence, "", "failed"])
                else:
                    manifest_rows.append([sec, idx, s_no, sentence, "", "failed"])

            pbar.update(1)

pbar.close()

# Write manifest CSV
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(manifest_header)
    writer.writerows(manifest_rows)

print("Done. Audio saved under:", OUTPUT_ROOT.resolve())
print("Manifest:", MANIFEST_CSV.resolve())
print("Tip: to improve pronunciation for a specific sentence, add a mapping to `pronunciation_overrides` and re-run.")


Generating sentence audio:   0%|          | 0/14 [00:00<?, ?it/s]

Generating sentence audio: 100%|██████████| 14/14 [00:18<00:00,  1.35s/it]

Done. Audio saved under: C:\Users\94772\Desktop\HearingProject\hearing_project_v0.01\sentence_audio
Manifest: C:\Users\94772\Desktop\HearingProject\hearing_project_v0.01\sentence_audio\manifest.csv
Tip: to improve pronunciation for a specific sentence, add a mapping to `pronunciation_overrides` and re-run.


In [36]:
# SINGLE NOTEBOOK CELL — produce per-index JSON manifest linking sentences -> audio paths
# plus per-highlighted-word vowel information
import json, csv, unicodedata, re, os
from pathlib import Path
from typing import List
from unidecode import unidecode

# ---------- Config (change names if you want) ----------
IN_JSON = Path("sentence_from_docx_notebook_with_highlights.json")
if not IN_JSON.exists():
    IN_JSON = Path("sentence_from_docx_notebook.json")  # fallback
if not IN_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {IN_JSON.resolve()}")

AUDIO_BASE_PATH = Path("hearing_project_v0.01/sentence_audio")  # base audio folder
OUT_JSON = Path("sentence_audio_manifest.json")
OUT_CSV = Path("sentence_audio_manifest.csv")

# ---------- Sinhala vowel/consonant maps (used for vowel extraction & transliteration) ----------
def normalize_text(s: str) -> str:
    return unicodedata.normalize("NFC", str(s) if s is not None else "")

CONS = {"ක":"k","ඛ":"kh","ග":"g","ඝ":"gh","ඟ":"ng","ච":"ch","ඡ":"chh","ජ":"j","ඣ":"jh","ඤ":"ny",
        "ට":"t","ඨ":"th","ඩ":"d","ඪ":"dh","ණ":"n","ත":"t","ථ":"th","ද":"d","ධ":"dh","න":"n",
        "ප":"p","ඵ":"ph","බ":"b","භ":"bh","ම":"m","ය":"y","ර":"r","ල":"l","ව":"v","ශ":"sh",
        "ෂ":"sh","ස":"s","හ":"h","ළ":"l","ෆ":"f","ඥ":"gn","ඦ":"gn"}

INDEP_V = {"අ":"a","ආ":"aa","ඇ":"ae","ඈ":"aae","ඉ":"i","ඊ":"ii","උ":"u","ඌ":"uu",
           "එ":"e","ඒ":"ee","ඔ":"o","ඕ":"oo","ඓ":"ai","ඖ":"au"}

V_SIGN = {"ා":"aa","ි":"i","ී":"ii","ු":"u","ූ":"uu","ෙ":"e","ේ":"ee","ෛ":"ai","ො":"o","ෝ":"oo","ෞ":"au","ෘ":"ru","ෲ":"ruu"}
DIAC = {"ං":"n","ඃ":"h"}
VIRAMA = "්"

INDEP_SET = set(INDEP_V.keys()); CONS_SET = set(CONS.keys()); V_SIGN_SET = set(V_SIGN.keys()); DIAC_SET = set(DIAC.keys())

# ---------- vowel extraction for the FIRST Sinhala letter/syllable ----------
def first_letter_vowel(word: str):
    """
    Return (vowel_char, vowel_name) for the FIRST Sinhala letter/syllable of `word`.
    - Independent vowel -> its mapping
    - Consonant followed by vowel-sign -> mapped independent vowel & short name
    - Consonant + virama or no sign -> implicit 'අ' ('a')
    - Non-sinhala -> ("","")
    """
    w = normalize_text(str(word)).strip()
    if not w:
        return "", ""
    first = w[0]
    # independent vowel
    if first in INDEP_SET:
        return first, INDEP_V[first]
    # Sinhala consonant block
    if '\u0D80' <= first <= '\u0DFF':
        if len(w) >= 2:
            nxt = w[1]
            if nxt in V_SIGN_SET:
                # map vowel sign -> independent-like char and short name
                # find the independent char by reverse mapping (best-effort)
                # we can't always map to the exact independent vowel char codepoint, so return the short name and the short independent name text where possible
                # Build a mapping from V_SIGN to an "independent char" approximate
                sign_to_indep = {
                    "ා":"ආ","ි":"ඉ","ී":"ඊ","ු":"උ","ූ":"ඌ","ෙ":"එ","ේ":"ඒ",
                    "ෛ":"ඓ","ො":"ඔ","ෝ":"ඕ","ෞ":"ඖ","ෘ":"උ","ෲ":"ඌ"
                }
                indep = sign_to_indep.get(nxt, "")
                name = V_SIGN.get(nxt, "")
                return indep, name
            if nxt == VIRAMA:
                return "අ", "a"
        return "අ", "a"
    # not Sinhala
    return "", ""

# ---------- transliteration (compact) ----------
def transliterate_text(s: str, cap_first: bool = False) -> str:
    w = normalize_text(s)
    i = 0; n = len(w); out = []
    while i < n:
        ch = w[i]
        if ch in INDEP_SET:
            out.append(INDEP_V[ch]); i += 1; continue
        if ch in CONS_SET:
            # gemination
            if i+2 < n and w[i+1] == VIRAMA and w[i+2] == ch:
                base = CONS[ch]; i += 3
                vowel_part = ""
                while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                    if w[i] in V_SIGN_SET: vowel_part = V_SIGN[w[i]]
                    elif w[i] in DIAC_SET: vowel_part += DIAC[w[i]]
                    i += 1
                if vowel_part == "": vowel_part = "a"
                out.append(base + base + vowel_part); continue
            base = CONS[ch]; i += 1
            if i < n and w[i] == VIRAMA:
                i += 1; out.append(base); continue
            vowel_part = ""
            while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                if w[i] in V_SIGN_SET: vowel_part = V_SIGN[w[i]]
                elif w[i] in DIAC_SET: vowel_part += DIAC[w[i]]
                i += 1
            if vowel_part == "": vowel_part = "a"
            out.append(base + vowel_part); continue
        if ch in V_SIGN_SET:
            out.append(V_SIGN[ch]); i += 1; continue
        if ch in DIAC_SET:
            out.append(DIAC[ch]); i += 1; continue
        out.append(ch); i += 1
    res = "".join(out)
    if cap_first and res:
        res = res[0].upper() + res[1:]
    return res

# ---------- filename helpers ----------
def safe_unicode_filename(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"[\\/:\*\?\"<>\|]", "_", s)
    s = re.sub(r"\s+", "_", s)
    return s[:60]

def ascii_snippet(s: str, maxlen=30) -> str:
    sn = unidecode(str(s) or "")
    sn = re.sub(r"[^\w\s\-]", "", sn).strip().replace(" ", "_")
    return (sn[:maxlen] or "sentence").strip("_")

def expected_sentence_filename(idx: str, s_no: int, sentence: str) -> str:
    idx2 = str(idx).zfill(2)
    snippet = ascii_snippet(sentence, maxlen=30)
    return f"{idx2}_{s_no}_{snippet}.mp3"

def get_portable_audio_path_for_sentence(section: str, idx: str, s_no: int, sentence: str) -> str:
    section_folder = AUDIO_BASE_PATH / safe_unicode_filename(section)
    fname = expected_sentence_filename(idx, s_no, sentence)
    candidate = section_folder / fname
    if candidate.exists():
        try:
            rel = os.path.relpath(candidate, start=Path.cwd()).replace("\\", "/").lstrip("./")
            if rel.startswith(AUDIO_BASE_PATH.name):
                return rel
            else:
                return f"{AUDIO_BASE_PATH.name}/{safe_unicode_filename(section)}/{fname}"
        except Exception:
            return f"{AUDIO_BASE_PATH.name}/{safe_unicode_filename(section)}/{fname}"
    return f"{AUDIO_BASE_PATH.name}/{safe_unicode_filename(section)}/{fname}"

# ---------- load input and build manifest entries ----------
raw = json.loads(IN_JSON.read_text(encoding="utf-8"))

manifest = []  # list of dicts, one per index/item

for section, payload in raw.items():
    for it in payload.get("items", []):
        idx = str(it.get("index", "")).zfill(2)
        # sentences list (preserve order)
        sentences = it.get("lines") or it.get("sentences") or []
        sentences = [str(x).strip() for x in sentences if str(x).strip()]
        if not sentences:
            continue
        # highlighted words from run-level parsing
        highlights = it.get("highlighted_words") or it.get("words") or []
        highlights = [str(h) for h in highlights][:2]  # keep at most 2

        # transliterate full sentences and highlighted words
        sentences_translit = [transliterate_text(s, cap_first=True) for s in sentences]
        highlights_translit = [transliterate_text(h, cap_first=True) for h in highlights]

        # compute audio paths per sentence (1-based sentence numbers)
        audio_paths = [get_portable_audio_path_for_sentence(section, idx, s_no+1, sentences[s_no]) for s_no in range(len(sentences))]

        # compute vowel info for each highlighted word
        highlighted_vowels = []
        for h in highlights:
            vch, vname = first_letter_vowel(h)
            highlighted_vowels.append({
                "word": h,
                "singlish": transliterate_text(h, cap_first=True),
                "main_vowel_char": vch,
                "main_vowel_name": vname
            })

        entry = {
            "section": section,
            "index": idx,
            "sentences": sentences,
            "sentences_translit": sentences_translit,
            "highlighted_words": highlights,
            "highlighted_translit": highlights_translit,
            "highlighted_vowels": highlighted_vowels,    # <-- new: vowel info per highlight
            "audio_paths": audio_paths
        }
        manifest.append(entry)

# ---------- write outputs ----------
OUT_JSON.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# also write CSV for quick inspection (one row per index; sentences joined by " || ")
with OUT_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    header = ["section","index","sentences","sentences_translit","highlighted_words","highlighted_translit","highlighted_vowels","audio_paths"]
    writer.writerow(header)
    for e in manifest:
        hv = "; ".join([f"{d['word']}|{d['main_vowel_char']}|{d['main_vowel_name']}" for d in e["highlighted_vowels"]])
        writer.writerow([
            e["section"],
            e["index"],
            " || ".join(e["sentences"]),
            " || ".join(e["sentences_translit"]),
            " | ".join(e["highlighted_words"]),
            " | ".join(e["highlighted_translit"]),
            hv,
            " | ".join(e["audio_paths"])
        ])

print("Wrote manifest JSON:", OUT_JSON.resolve())
print("Wrote manifest CSV:", OUT_CSV.resolve())
print("Entries:", len(manifest))

# show first 3 entries for quick preview
for m in manifest[:3]:
    print(json.dumps(m, ensure_ascii=False, indent=2))


Wrote manifest JSON: C:\Users\94772\Desktop\HearingProject\sentence_audio_manifest.json
Wrote manifest CSV: C:\Users\94772\Desktop\HearingProject\sentence_audio_manifest.csv
Entries: 7
{
  "section": "NoHeader",
  "index": "01",
  "sentences": [
    "ඔහු ගසේ මල කැඩුවා",
    "ඔහු ගසේ මුල කැඩුවා"
  ],
  "sentences_translit": [
    "Ohu gasee mala kaැduvaa",
    "Ohu gasee mula kaැduvaa"
  ],
  "highlighted_words": [
    "මල",
    "මුල"
  ],
  "highlighted_translit": [
    "Mala",
    "Mula"
  ],
  "highlighted_vowels": [
    {
      "word": "මල",
      "singlish": "Mala",
      "main_vowel_char": "අ",
      "main_vowel_name": "a"
    },
    {
      "word": "මුල",
      "singlish": "Mula",
      "main_vowel_char": "උ",
      "main_vowel_name": "u"
    }
  ],
  "audio_paths": [
    "sentence_audio/NoHeader/01_1_ohu_gsee_ml_kaedduvaa.mp3",
    "sentence_audio/NoHeader/01_2_ohu_gsee_mul_kaedduvaa.mp3"
  ]
}
{
  "section": "NoHeader",
  "index": "02",
  "sentences": [
    "නංගී කජු කන්න ආසයි"